This works on the parquet file from the 02-11-2026 version of https://www.fema.gov/openfema-data-page/fima-nfip-redacted-policies-v2

In [1]:
import duckdb

In [2]:
con = duckdb.connect(database=':memory:')
con.execute("SET memory_limit = '2GB';")
con.execute("SET threads = 2;")

# Load parquet file
con.execute("CREATE OR REPLACE VIEW nfip AS SELECT * FROM read_parquet('FimaNfipPoliciesV2.parquet')")

In [3]:
# Representative sample: hash-based deterministic sampling (~1% train, ~1% test)
# No sort - streams through data; reproducible as long as parquet is unchanged
con.execute("""
    CREATE OR REPLACE VIEW indexed AS
    SELECT *, row_number() OVER () AS idx
    FROM read_parquet('FimaNfipPoliciesV2.parquet')
""")

In [4]:
# Train: hash(idx+42) % 200 = 0 (~0.5%), Test: hash(idx+42) % 200 = 1 (~0.5%)
# Exact row count depends on dataset size; representative across full date range
con.execute("""
    CREATE OR REPLACE VIEW train_sample AS
    SELECT * EXCLUDE (idx) FROM indexed WHERE hash(idx + 42) % 200 = 0
""")
con.execute("""
    CREATE OR REPLACE VIEW test_sample AS
    SELECT * EXCLUDE (idx) FROM indexed WHERE hash(idx + 42) % 200 = 1
""")

In [ ]:
# Export training sample (representative across full dataset)
con.execute("""
    COPY (SELECT * FROM train_sample) TO 'sample_train.parquet' (FORMAT PARQUET)
""")

In [ ]:
# Export testing sample (representative across full dataset)
con.execute("""
    COPY (SELECT * FROM test_sample) TO 'sample_test.parquet' (FORMAT PARQUET)
""")

In [ ]:
con.execute("""
    SELECT 
        count(policyEffectiveDate) AS count,
        min(policyEffectiveDate) AS min,
        max(policyEffectiveDate) AS max,
        max(policyEffectiveDate) - min(policyEffectiveDate) AS range,
        quantile_cont(policyEffectiveDate, 0.5) AS median
    FROM read_parquet('sample_train.parquet')
""").df()

,count,min,max,range,median
0,356839,2009-01-01,2025-11-01,6148,2016-10-17


In [ ]:
con.close()